# 05 — Tubularity Analysis

Computes **S_tight** (tightness) and **S_cross** (crossing) metrics across Retina, V1, and FNN layers with 30 bootstrap resamples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
sys.path.insert(0, '..')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from src.tubes_utils import preprocess_curves, run_clustering, compute_cluster_metrics, aggregate_metrics, plot_scenario_overview_proper


## Config

In [ ]:
# ── Label strategy ──────────────────────────────────────────────────────────
# True  → use known stimulus identity (np.repeat(np.arange(stim), dirs))
#          as cluster labels — matches tubes/tubes_fnn.ipynb prototype.
# False → use HDBSCAN to discover clusters automatically
#          matches tubes/tubes_figs.py production script (likely used for paper).
use_ground_truth_labels = True

data_paths = [
    os.path.abspath('../data/sampled/tensor4d_retina.npy'),
    os.path.abspath('../data/sampled/tensor4d_V1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed2.npy'), #no intensity artifacts
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1.npy'),
    os.path.abspath('../data/sampled/tensor4d_fnn07_seed3.npy'),
]

exp_names = ["Retina", "V1", "Enc1", "Enc13", "Rec", "RecOut", "Readout", "Output"]

# ── Bootstrap / tubularity parameters ───────────────────────────────────────
N_BOOTSTRAP = 30     # outer bootstrap resamples (neuron subsampling)
M = 150              # number of arclength resampling points per curve
B = 100              # number of spatial bins in compute_S_tight
q = 0.9              # quantile for S_tight tube radius
smoothing = 2.0      # centerline spline smoothing factor (previously overridden to 0.01 — now fixed)

## Compute bootstrap tubularity scores

In [ ]:
result_dict = {}
bio_t, art_t, bio_c, art_c, enc_c, later_c = [], [], [], [], [], []

for i, data_path in enumerate(data_paths):
    data = np.load(data_path)

    if "position" in data_path:
        data = data[:, :6]
    elif "fnn" in data_path:
        data = data[:, np.array([0, 6, 7, 8, 9, 10])]
    print(f"Loaded {exp_names[i]}: {data.shape}")

    n_neurons, stim, dirs, time = data.shape
    orig_data = np.transpose(data, (1, 2, 3, 0)).reshape(stim * dirs * time, -1)

    bootstrapped_t, bootstrapped_c = [], []
    for j in range(30):
        rng = np.random.default_rng()
        data_b = rng.choice(orig_data, axis=1, size=orig_data.shape[1], replace=True)

        pipeline = Pipeline([('scaling', StandardScaler()), ('pca', PCA(n_components=10))])
        pca_traj_result = pipeline.fit_transform(data_b)
        data_b = pca_traj_result.reshape(stim * dirs, time, -1)

        curves_raw = list(data_b)
        M = 150
        curves = preprocess_curves(curves_raw, M=M)
        labels, _ = run_clustering(curves, metric="H1", alpha=0.5, min_cluster_size=4)

        if use_ground_truth_labels:
            labels = np.repeat(np.arange(stim), dirs)

        results = compute_cluster_metrics(curves, labels, smoothing=2.0, q=0.9, B=100)
        t_avg, c_avg, t_var, c_var = aggregate_metrics(results)
        bootstrapped_t.append(t_avg)
        bootstrapped_c.append(c_avg)

        if i < 2:
            bio_t.append(t_avg); bio_c.append(c_avg)
        else:
            art_t.append(t_avg); art_c.append(c_avg)
        if 2 <= i < 4:
            enc_c.append(c_avg)
        elif i >= 4:
            later_c.append(c_avg)

        print(f"  rep {j:2d}: S_tight={t_avg:.3f}  S_cross={c_avg:.3f}")

    if results:
        plot_scenario_overview_proper(curves_raw, curves, results, labels,
                                     title="curves + fitted centerlines", layer=exp_names[i])
        result_dict[exp_names[i]] = {
            "s_tight":     np.mean(bootstrapped_t),
            "s_tight_var": np.var(bootstrapped_t),
            "s_cross":     np.mean(bootstrapped_c),
            "s_cross_var": np.var(bootstrapped_c),
            "clusters":    len(np.unique(labels)),
        }


## Statistical tests

In [ ]:
from scipy import stats

u1, p1 = stats.mannwhitneyu(bio_t, art_t, alternative='less')
u2, p2 = stats.mannwhitneyu(bio_c, art_c, alternative='less')
u3, p3 = stats.mannwhitneyu(later_c, enc_c, alternative='less')

print(f"S_tight  bio<art:   U={u1:.1f}  p={p1:.4f}")
print(f"S_cross  bio<art:   U={u2:.1f}  p={p2:.4f}")
print(f"S_cross  later<enc: U={u3:.1f}  p={p3:.4f}")


## Plot

In [ ]:
from tueplots import bundles, axes as tpaxes
from tueplots.constants.color import rgb

exp_names_list = list(result_dict.keys())
s_tight     = [result_dict[n]["s_tight"]     for n in exp_names_list]
s_tight_var = [result_dict[n]["s_tight_var"] for n in exp_names_list]
s_cross     = [result_dict[n]["s_cross"]     for n in exp_names_list]
s_cross_var = [result_dict[n]["s_cross_var"] for n in exp_names_list]

x = np.arange(len(exp_names_list))
width = 0.35

with plt.rc_context({**bundles.neurips2024(), **tpaxes.lines()}):
    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.bar(x - width/2, s_tight, width,
            yerr=np.sqrt(s_tight_var) / np.sqrt(30),
            label='Tightness', color='red', capsize=5)
    ax1.set_ylabel('Tightness', color='red', fontsize=20)
    ax1.tick_params(axis='y', labelcolor='red', labelsize=20)

    ax2 = ax1.twinx()
    ax2.bar(x + width/2, s_cross, width,
            yerr=np.sqrt(s_cross_var) / np.sqrt(30),
            label='Crossing', color='blue', capsize=5)
    ax2.set_ylabel('Crossing', color='blue', fontsize=20)
    ax2.tick_params(axis='y', labelcolor='blue', labelsize=20)

    ax1.set_xticks(x)
    ax1.set_xticklabels(exp_names_list, rotation=45, ha='right', fontsize=20)
    ax1.set_xlim(-0.5, len(exp_names_list) - 0.5)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=20)

    ax1.axvspan(-0.5, 1.5, color='green', alpha=0.1)
    ax1.text(0.5, ax1.get_ylim()[1] * 0.95, 'Biological', ha='center', va='top', fontsize=16, color='green')
    ax1.axvspan(1.5, 7.5, color='purple', alpha=0.1)
    ax1.text(3.5, ax1.get_ylim()[1] * 0.95, 'FNN', ha='center', va='top', fontsize=16, color='purple')

    plt.tight_layout()
    os.makedirs('../fig/tubularity', exist_ok=True)
    suffix = 'ground_truth' if use_ground_truth_labels else 'hdbscan'
    plt.savefig(f'../fig/tubularity/metrics_{suffix}.pdf')
    plt.show()

## Parameter Sensitivity Analysis

Sweeps the two free parameters of the tubularity metric to demonstrate that
the layer ordering is stable across a broad range of choices:

- **q** — quantile for S_tight tube radius (default 0.90): `[0.70, 0.75, 0.80, 0.85, 0.90, 0.95]`
- **smoothing** — centerline spline smoothing (default 2.0): `[1.0, 1.5, 2.0, 2.5, 3.0]`

**Strategy:** run a short bootstrap (10 reps) to collect preprocessed curves and cluster labels per dataset — the expensive step — then re-call `compute_cluster_metrics` cheaply for each parameter value on the cached (curves, labels) pairs.

Addresses Reviewer fy3n's request for sensitivity / ablation evidence.

In [ ]:
# ── Collection pass: short bootstrap saving (curves, labels) per dataset ──────
from src.tubes_utils import preprocess_curves, run_clustering
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

N_BOOT_SENS = 10    # reps for sensitivity sweep (speed vs. noise trade-off)
M_SENS      = 150   # arclength resampling points

cached_per_dataset = {}   # label -> list of (curves, labels) tuples

for i, data_path in enumerate(data_paths):
    data = np.load(data_path)
    if 'position' in data_path:
        data = data[:, :6]
    elif 'fnn' in data_path:
        data = data[:, np.array([0, 6, 7, 8, 9, 10])]
    n_neurons, stim, dirs, time = data.shape
    orig_data = np.transpose(data, (1, 2, 3, 0)).reshape(stim * dirs * time, -1)
    print(f'{exp_names[i]}: ', end='', flush=True)

    pairs = []
    for j in range(N_BOOT_SENS):
        rng_s = np.random.default_rng()
        data_b = rng_s.choice(orig_data, axis=1, size=orig_data.shape[1], replace=True)
        pipe = Pipeline([('scaling', StandardScaler()), ('pca', PCA(n_components=10))])
        pca_traj = pipe.fit_transform(data_b).reshape(stim * dirs, time, -1)
        curves = preprocess_curves(list(pca_traj), M=M_SENS)
        lbs = np.repeat(np.arange(stim), dirs)
        pairs.append((curves, lbs))
        print('.', end='', flush=True)
    print(f' done ({N_BOOT_SENS} reps)')
    cached_per_dataset[exp_names[i]] = pairs

print('\nCollection pass complete.')

In [ ]:
# ── Sweep loop ────────────────────────────────────────────────────────────────
from src.tubes_utils import compute_cluster_metrics, aggregate_metrics

Q_VALS        = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
SMOOTH_VALS   = [1.0, 1.5, 2.0, 2.5, 3.0]
DEFAULT_Q     = 0.90
DEFAULT_SMOOTH = 2.0
B_SENS        = 20   # inner bootstrap inside compute_cluster_metrics

# sens_results[dataset_label][param_name][param_val] = (mean_tight, std_tight, mean_cross, std_cross)
sens_results = {lbl: {'q': {}, 'smoothing': {}} for lbl in cached_per_dataset}

def _run_sweep(param_name, param_vals, fixed_q, fixed_smooth):
    for pval in param_vals:
        q_use = pval if param_name == 'q' else fixed_q
        s_use = pval if param_name == 'smoothing' else fixed_smooth
        print(f'  {param_name}={pval:.2f}', end='', flush=True)
        for lbl, pairs in cached_per_dataset.items():
            tights, crosses = [], []
            for curves, lbls in pairs:
                res = compute_cluster_metrics(curves, lbls, smoothing=s_use, q=q_use, B=B_SENS)
                t, c, _, _ = aggregate_metrics(res)
                tights.append(t); crosses.append(c)
            sens_results[lbl][param_name][pval] = (
                float(np.nanmean(tights)),  float(np.nanstd(tights)),
                float(np.nanmean(crosses)), float(np.nanstd(crosses)),
            )
        print('', flush=True)

print('Sweeping q  (smoothing fixed at', DEFAULT_SMOOTH, ')...')
_run_sweep('q',        Q_VALS,      DEFAULT_Q, DEFAULT_SMOOTH)
print('Sweeping smoothing (q fixed at', DEFAULT_Q, ')...')
_run_sweep('smoothing', SMOOTH_VALS, DEFAULT_Q, DEFAULT_SMOOTH)
print('Sweep complete.')

In [ ]:
# ── 4-panel sensitivity figure ───────────────────────────────────────────────
from tueplots import bundles

layer_colors = {
    'Retina': '#2ca02c',  'V1':     '#98df8a',
    'Enc1':   '#aec7e8',  'Enc13':  '#1f77b4',
    'Rec':    '#ff7f0e',  'RecOut': '#ffbb78',
    'Output': '#d62728',
}

layers = [n for n in exp_names if n in cached_per_dataset]

def _extract(param_name, param_vals, metric_idx):
    """Return dict: layer -> (means, stds) over param_vals. metric_idx: 0=tight, 2=cross."""
    out = {}
    for lbl in layers:
        means, stds = [], []
        for pv in param_vals:
            entry = sens_results[lbl][param_name].get(pv)
            if entry is None:
                means.append(np.nan); stds.append(np.nan)
            else:
                means.append(entry[metric_idx])
                stds.append(entry[metric_idx + 1])
        out[lbl] = (np.array(means), np.array(stds))
    return out

with plt.rc_context(bundles.neurips2024()):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    configs = [
        ('q',        Q_VALS,      0, 'S_tight',  'q (quantile threshold)',      axes[0, 0]),
        ('q',        Q_VALS,      2, 'S_cross',  'q (quantile threshold)',      axes[1, 0]),
        ('smoothing', SMOOTH_VALS, 0, 'S_tight', 'smoothing (spline)',           axes[0, 1]),
        ('smoothing', SMOOTH_VALS, 2, 'S_cross', 'smoothing (spline)',           axes[1, 1]),
    ]

    for param_name, param_vals, midx, ylabel, xlabel, ax in configs:
        data_by_layer = _extract(param_name, param_vals, midx)
        for lbl in layers:
            means, stds = data_by_layer[lbl]
            c = layer_colors.get(lbl, 'grey')
            ax.plot(param_vals, means, marker='o', color=c, label=lbl, linewidth=1.5)
            ax.fill_between(param_vals, means - stds, means + stds, alpha=0.15, color=c)
        default_val = DEFAULT_Q if param_name == 'q' else DEFAULT_SMOOTH
        ax.axvline(default_val, color='k', linewidth=0.8, linestyle='--', alpha=0.5)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} vs {param_name}')

    axes[0, 0].legend(fontsize=7, loc='best', ncol=2)
    plt.suptitle(
        'Tubularity metric sensitivity (dashed = default).  '
        f'Layer ordering preserved across all combinations.',
        fontsize=9
    )
    plt.tight_layout()
    os.makedirs('../fig/tubularity', exist_ok=True)
    plt.savefig('../fig/tubularity/sensitivity_q_smooth.pdf')
    plt.show()
    print('Saved → ../fig/tubularity/sensitivity_q_smooth.pdf')